### Setup + harmonisation

In [1]:
import pandas as pd
import numpy as np
from datetime import date

date_then = date(2025, 7, 1)

register = pd.read_excel(r"..\data\public_data_registerF.xlsx")
survey   = pd.read_excel(r"..\data\private_dataF.xlsx")

# harmonise
register = register.rename(columns={"last_voted": "evote"})
register = register[register["evote"].isin([0, 1])].copy()

def prepare(df, ref=date_then):
    df = df.copy()

    # types
    df["dob"] = pd.to_datetime(df["dob"], format="%d/%m/%Y", errors="coerce")
    df["zip"] = df["zip"].astype(str)
    df["evote"] = df["evote"].astype(int)

    # age -> age group
    df["age"] = df["dob"].apply(
        lambda d: ref.year - d.year - ((ref.month, ref.day) < (d.month, d.day))
    )
    df["age_group"] = pd.cut(
        df["age"],
        bins=[18, 30, 45, 65, np.inf],
        labels=["18-29", "30-44", "45-64", "65 or older"],
        right=False,
    )

    # citizenship -> Danish / non-Danish
    df["citizenship"] = df["citizenship"].apply(
        lambda x: "Danish" if str(x).strip() == "Denmark" else "non-Danish"
    )

    return df

survey   = prepare(survey)
register = prepare(register)

qi = ["sex", "age_group", "zip", "marital_status", "citizenship", "evote"]

### Frequencies and individual risk

In [2]:
F_tab = register.groupby(qi).size().rename("F").reset_index()   # population count
f_tab = survey.groupby(qi).size().rename("f").reset_index()     # sample count

risk = (survey
        .merge(F_tab, on=qi, how="left")
        .merge(f_tab, on=qi, how="left"))
risk["r"] = 1 / risk["F"]

# survey records with no match in the register (should be ~0; investigate otherwise)
print("unmatched:", risk["F"].isna().sum())

unmatched: 0


C:\Users\anial\AppData\Local\Temp\ipykernel_3992\235080015.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  F_tab = register.groupby(qi).size().rename("F").reset_index()   # population count
C:\Users\anial\AppData\Local\Temp\ipykernel_3992\235080015.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  f_tab = survey.groupby(qi).size().rename("f").reset_index()     # sample count


### Summary metrics

In [3]:
def summarise(risk):
    r, F, f = risk["r"], risk["F"], risk["f"]
    return pd.Series({
        "n":                 len(risk),
        "sample_uniques":    (f == 1).mean(),
        "population_unique": (F == 1).mean(),
        "mean_risk":         r.mean(),
        "max_risk":          r.max(),
        "share_r_gt_0.5":    (r > 0.5).mean(),
        "share_r_gt_0.2":    (r > 0.2).mean(),
        "share_r_gt_0.1":    (r > 0.1).mean(),
        "min_k":             F.min(),
        "viol_k2":           (F < 2).sum(),
        "viol_k5":           (F < 5).sum(),
        "viol_k10":          (F < 10).sum(),
    })

print(summarise(risk))

n                    200.000000
sample_uniques         0.335000
population_unique      0.085000
mean_risk              0.229907
max_risk               1.000000
share_r_gt_0.5         0.085000
share_r_gt_0.2         0.285000
share_r_gt_0.1         0.545000
min_k                  1.000000
viol_k2               17.000000
viol_k5               57.000000
viol_k10             109.000000
dtype: float64


### Attribute disclosure on party

In [4]:
# l-diversity of party within each QI group (sample level)
risk["n_parties"] = risk.groupby(qi)["party"].transform("nunique")
print("share in groups homogeneous in party:", (risk["n_parties"] == 1).mean())

# records with F == 1 have their party disclosed with certainty
print("party certainly disclosed (F==1):", (risk["F"] == 1).mean())

share in groups homogeneous in party: 0.555
party certainly disclosed (F==1): 0.085


C:\Users\anial\AppData\Local\Temp\ipykernel_3992\1302531292.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  risk["n_parties"] = risk.groupby(qi)["party"].transform("nunique")


### Check with the real names

In [5]:
attack = (survey[qi + ["name", "party"]].rename(columns={"name": "true_name"})
          .merge(register[qi + ["name"]].rename(columns={"name": "reg_name"}), on=qi))

# keep respondents with exactly one candidate in the register
n_cand = attack.groupby("true_name")["reg_name"].transform("size")
unique = attack[n_cand == 1]
print("unique matches:", len(unique))
print("correct among unique matches:", (unique["true_name"] == unique["reg_name"]).mean())

# overall expected success if the attacker guesses randomly among the F candidates
# equals risk["r"].mean()

unique matches: 17
correct among unique matches: 1.0


### Resusable function for later

In [6]:
def risk_report(survey, register, qi):
    F_tab = register.groupby(qi, dropna=False).size().rename("F").reset_index()
    f_tab = survey.groupby(qi, dropna=False).size().rename("f").reset_index()
    out = survey.merge(F_tab, on=qi, how="left").merge(f_tab, on=qi, how="left")
    out["r"] = 1 / out["F"]
    return out, summarise(out)